# **Deskripsi Dataset**

Dataset yang digunakan merupakan gambaran tentang mahasiswa yang terdaftar dalam 
berbagai program sarjana yang ditawarkan oleh perguruan tinggi. Data ini mencakup data 
demografis, faktor sosial-ekonomi, dan informasi kinerja akademik yang dapat digunakan 
untuk menganalisis faktor-faktor yang mungkin memprediksi tingkat kesuksesan akademik 
mahasiswa.

# **Requirements and Config**

In [11]:
# !pip install seaborn matplotlib numpy pandas

In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
sys.path.append(os.getcwd())

from allyouneed.tree import DecisionTreeClassifier

pd.set_option('display.max_columns', None)

class settings:
    DATA_DIR = "../dataset/"
    TRAIN_FILE = "train.csv"
    TEST_FILE = "test.csv"
    SUBMISSION_FILE = "sample_submission.csv"
    INDEX_COL = "Student_ID"
    SEED = 42
    TARGET = "Target"
    TRAIN_PATH = DATA_DIR + TRAIN_FILE
    TEST_PATH = DATA_DIR + TEST_FILE
    SUBMISSION_PATH = DATA_DIR + SUBMISSION_FILE

In [13]:
train = pd.read_csv(settings.TRAIN_PATH, index_col=settings.INDEX_COL)
test = pd.read_csv(settings.TEST_PATH, index_col=settings.INDEX_COL)
submission = pd.read_csv(settings.SUBMISSION_PATH, index_col=settings.INDEX_COL)
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3096 entries, 3743 to 3303
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  3096 non-null   int64  
 1   Application mode                                3096 non-null   int64  
 2   Application order                               3096 non-null   int64  
 3   Course                                          3096 non-null   int64  
 4   Daytime/evening attendance	                     3096 non-null   int64  
 5   Previous qualification                          3096 non-null   int64  
 6   Previous qualification (grade)                  3096 non-null   float64
 7   Nacionality                                     3096 non-null   int64  
 8   Mother's qualification                          3096 non-null   int64  
 9   Father's qualification                     

# **Exploratory Data Analysis**

# **Preprocessing**

## **Data Cleaning**

In [14]:
def drop_columns(df):
    col = []
    return df.drop(columns=col)

## **Data Transformation**

In [15]:
import numpy as np
import pandas as pd

class OneHotEncoder:
    def __init__(self, columns=None, handle_unknown="ignore", dtype=float):
        self.columns = columns
        self.handle_unknown = handle_unknown
        self.dtype = dtype
        self.categories_ = {}   # dict: col -> list(categories)
        self.feature_names_ = []
        self.fitted = False

    def fit(self, X):
        X = self._to_dataframe(X)

        if self.columns is None:
            self.columns = X.select_dtypes(include=["object", "category"]).columns.tolist()

        for col in self.columns:
            self.categories_[col] = np.unique(X[col].astype(str)).tolist()

        self.feature_names_ = []
        for col in self.columns:
            for cat in self.categories_[col]:
                self.feature_names_.append(f"{col}__{cat}")

        self.fitted = True
        return self

    def transform(self, X):
        self._check_is_fitted()
        X = self._to_dataframe(X)

        encoded_arrays = []

        for col in self.columns:
            values = X[col].astype(str).values
            categories = self.categories_[col]

            mat = np.zeros((len(values), len(categories)), dtype=self.dtype)

            for i, val in enumerate(values):
                if val in categories:
                    j = categories.index(val)
                    mat[i, j] = 1.0
                else:
                    if self.handle_unknown == "error":
                        raise ValueError(f"Unknown category '{val}' in column '{col}'")

            encoded_arrays.append(mat)

        if encoded_arrays:
            return np.hstack(encoded_arrays).astype(self.dtype)
        else:
            return np.zeros((len(X), 0), dtype=self.dtype)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, encoded):
        self._check_is_fitted()
        encoded = np.asarray(encoded)

        idx = 0
        outputs = {}

        for col in self.columns:
            num_cats = len(self.categories_[col])
            slice_col = encoded[:, idx: idx + num_cats]
            idx += num_cats

            inv = []
            for row in slice_col:
                if np.sum(row) == 0:
                    inv.append(None)  # tidak tahu kategori
                else:
                    j = np.argmax(row)
                    inv.append(self.categories_[col][j])

            outputs[col] = inv
        
        return pd.DataFrame(outputs)

    def _to_dataframe(self, X):
        if not isinstance(X, pd.DataFrame):
            return pd.DataFrame(X)
        return X

    def _check_is_fitted(self):
        if not self.fitted:
            raise ValueError("OneHotEncoder must be fitted before transform().")


In [16]:
class LabelEncoder:
    def __init__(self):
        self.class_to_int = {}
        self.int_to_class = {}

    def fit(self, y):
        classes = np.unique(y)
        self.class_to_int = {c: i for i, c in enumerate(classes)}
        self.int_to_class = {i: c for c, i in self.class_to_int.items()}

    def transform(self, y):
        return np.array([self.class_to_int[c] for c in y], dtype=int)

    def inverse_transform(self, y_int):
        return np.array([self.int_to_class[i] for i in y_int])
    
label_encoder = LabelEncoder()
label_encoder.fit(train[settings.TARGET])

## **Feature Selection**

## **Dimensionality Reduction**

## **Pipeline**

In [17]:
class Preprocessing:
    def __init__(self, categorical_cols=None):
        self.categorical_cols = categorical_cols
        self.ohe = OneHotEncoder(columns=categorical_cols, dtype=float)
        self.numeric_cols = None
        self.fitted = False

    def fit(self, X):
        X = self._to_dataframe(X)
        self.numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        self.ohe.fit(X)

        self.fitted = True
        return self

    def transform(self, X):
        self._check_is_fitted()
        X = self._to_dataframe(X)

        X_num = X[self.numeric_cols].astype(float).values

        X_cat = self.ohe.transform(X)

        # final matrix
        return np.hstack([X_num, X_cat]).astype(float)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def _to_dataframe(self, X):
        if not isinstance(X, pd.DataFrame):
            return pd.DataFrame(X)
        return X

    def _check_is_fitted(self):
        if not self.fitted:
            raise ValueError("Preprocessing must be fitted before transform().")


CATEGORICAL_COLS = [
    "Marital status",
    "Course",
    "Previous qualification",
]

preprocessor = Preprocessing(categorical_cols=CATEGORICAL_COLS)

# **Modeling and Validation**

## **Metrics**

In [18]:
from abc import ABC, abstractmethod

class Metric(ABC):
    """
    Metric interface: all metrics must implement __call__
    """

    @abstractmethod
    def __call__(self, y_true, y_pred):
        pass


# ============================================================
# Utility: Confusion Matrix
# ============================================================

class ConfusionMatrix:
    """
    Computes confusion matrix using NumPy.
    """

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        labels = np.unique(np.concatenate([y_true, y_pred]))
        label_to_idx = {label: idx for idx, label in enumerate(labels)}
        
        cm = np.zeros((len(labels), len(labels)), dtype=int)

        for t, p in zip(y_true, y_pred):
            cm[label_to_idx[t], label_to_idx[p]] += 1
        
        return cm, labels


# ============================================================
# Accuracy
# ============================================================

class Accuracy(Metric):
    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        return float(np.mean(y_true == y_pred))


# ============================================================
# Precision Macro
# ============================================================

class PrecisionMacro(Metric):
    """
    Macro-averaged precision (one-vs-rest)
    """

    def __init__(self, include_pred_only=False):
        self.include_pred_only = include_pred_only

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        if self.include_pred_only:
            labels = np.unique(np.concatenate([y_true, y_pred]))
        else:
            labels = np.unique(y_true)

        precisions = []

        for c in labels:
            tp = np.sum((y_true == c) & (y_pred == c))
            fp = np.sum((y_true != c) & (y_pred == c))

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            precisions.append(precision)

        return float(np.mean(precisions))


# ============================================================
# Recall Macro
# ============================================================

class RecallMacro(Metric):
    """
    Macro-averaged recall (one-vs-rest)
    """

    def __init__(self, include_pred_only=False):
        self.include_pred_only = include_pred_only

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        if self.include_pred_only:
            labels = np.unique(np.concatenate([y_true, y_pred]))
        else:
            labels = np.unique(y_true)

        recalls = []

        for c in labels:
            tp = np.sum((y_true == c) & (y_pred == c))
            fn = np.sum((y_true == c) & (y_pred != c))

            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            recalls.append(recall)

        return float(np.mean(recalls))


# ============================================================
# F1 Macro
# ============================================================

class F1Macro(Metric):
    def __init__(self, include_pred_only=False):
        self.include_pred_only = include_pred_only

    def __call__(self, y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        if self.include_pred_only:
            labels = np.unique(np.concatenate([y_true, y_pred]))
        else:
            labels = np.unique(y_true)

        f1_scores = []

        for c in labels:
            tp = np.sum((y_true == c) & (y_pred == c))
            fp = np.sum((y_pred == c) & (y_true != c))
            fn = np.sum((y_true == c) & (y_pred != c))

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0

            if precision + recall == 0:
                f1 = 0.0
            else:
                f1 = 2 * precision * recall / (precision + recall)

            f1_scores.append(f1)

        return float(np.mean(f1_scores))


# ============================================================
# Metric Collection (Run multiple metrics at once)
# ============================================================

class MetricCollection:
    """
    Utility to compute multiple metrics at once.
    """

    def __init__(self, metrics: dict):
        """
        metrics: dict[name -> Metric instance]
        """
        self.metrics = metrics

    def __call__(self, y_true, y_pred):
        return {name: metric(y_true, y_pred) for name, metric in self.metrics.items()}


## **Validation Splitter**

In [19]:
# ============================================================
# Base Class for K-Fold Variants
# ============================================================

class BaseKFold(ABC):

    def __init__(self, n_splits=5, shuffle=True, random_state=None):
        assert n_splits >= 2, "n_splits must be >= 2"
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state

    def _get_permutation(self, n_samples):
        if self.shuffle:
            rng = np.random.default_rng(self.random_state)
            return rng.permutation(n_samples)
        return np.arange(n_samples)

    @abstractmethod
    def split(self, X, y=None):
        pass


# ============================================================
# Standard K-Fold
# ============================================================
class KFold(BaseKFold):
    
    def split(self, X, y=None):
        X = np.asarray(X)
        n_samples = len(X)

        indices = self._get_permutation(n_samples)

        fold_sizes = np.full(self.n_splits, n_samples // self.n_splits, dtype=int)
        fold_sizes[: n_samples % self.n_splits] += 1

        current = 0
        for fold_size in fold_sizes:
            start, stop = current, current + fold_size
            test_index = indices[start:stop]
            train_index = np.concatenate((indices[:start], indices[stop:]))
            yield train_index, test_index
            current = stop


# ============================================================
# Stratified K-Fold
# ============================================================

class StratifiedKFold(BaseKFold):

    def split(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)

        indices = self._get_permutation(len(y))
        y_shuffled = y[indices]

        classes = np.unique(y_shuffled)
        class_indices = {cls: np.where(y_shuffled == cls)[0] for cls in classes}

        folds = [[] for _ in range(self.n_splits)]

        rng = np.random.default_rng(self.random_state)

        for cls in classes:
            cls_idx = class_indices[cls]

            if self.shuffle:
                cls_idx = rng.permutation(cls_idx)

            split_cls = np.array_split(cls_idx, self.n_splits)

            for i, batch in enumerate(split_cls):
                folds[i].extend(batch.tolist())

        for i in range(self.n_splits):
            test_local_idx = np.array(sorted(folds[i]))
            train_local_idx = np.array(sorted(list(set(range(len(y))) - set(test_local_idx))))

            yield indices[train_local_idx], indices[test_local_idx]


## **Modeling**

In [20]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=settings.SEED)

models = {
    'DT': DecisionTreeClassifier()
}

metrics = MetricCollection({
    'accuracy': Accuracy(),
    'precision_macro': PrecisionMacro(),
    'recall_macro': RecallMacro(),
    'f1_macro': F1Macro()
})

all_results = {}  # store aggregated results for each model

for name, model in models.items():
    print(f"Training model: {name}")
    fold_results = []  # store metric dict for each fold

    for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[settings.TARGET])):
        
        # -------------------------
        # Split data
        # -------------------------
        X_train = train.drop(columns=[settings.TARGET]).iloc[train_idx]
        y_train_raw = train[settings.TARGET].iloc[train_idx].values
        y_train = label_encoder.transform(y_train_raw)

        X_val = train.drop(columns=[settings.TARGET]).iloc[val_idx]
        y_val_raw = train[settings.TARGET].iloc[val_idx].values
        y_val = label_encoder.transform(y_val_raw)

        # -------------------------
        # Preprocessing
        # -------------------------
        train_data = preprocessor.fit_transform(X_train)
        val_data = preprocessor.transform(X_val)

        # -------------------------
        # Training
        # -------------------------
        model.fit(train_data, y_train)

        # -------------------------
        # Validation
        # -------------------------
        val_preds = model.predict(val_data)

        # -------------------------
        # Metrics
        # -------------------------
        results = metrics(y_val, val_preds)
        fold_results.append(results)

        print(f"Fold {fold + 1} results: {results}")

    # -------------------------
    # Aggregate across folds
    # -------------------------
    df_results = pd.DataFrame(fold_results)
    model_mean = df_results.mean().to_dict()

    all_results[name] = {
        "per_fold": fold_results,
        "mean": model_mean
    }

    print(f"\nAggregated results for model {name}:")
    print(df_results)
    print("Mean metrics:", model_mean)
    print("=" * 60)


Training model: DT


Fold 1 results: {'accuracy': 0.6892109500805152, 'precision_macro': 0.6224249636678127, 'recall_macro': 0.6246217647376667, 'f1_macro': 0.6232259897842173}
Fold 2 results: {'accuracy': 0.7011308562197092, 'precision_macro': 0.6448989931617305, 'recall_macro': 0.6394419984701489, 'f1_macro': 0.64179225524517}
Fold 3 results: {'accuracy': 0.6930533117932148, 'precision_macro': 0.6212762797171933, 'recall_macro': 0.6209040186207576, 'f1_macro': 0.6201348993086141}
Fold 4 results: {'accuracy': 0.6736672051696284, 'precision_macro': 0.61828540423223, 'recall_macro': 0.6188534731736176, 'f1_macro': 0.6167439939707416}
Fold 5 results: {'accuracy': 0.6796116504854369, 'precision_macro': 0.6193278865466038, 'recall_macro': 0.6161395287608881, 'f1_macro': 0.6175623103026274}

Aggregated results for model DT:
   accuracy  precision_macro  recall_macro  f1_macro
0  0.689211         0.622425      0.624622  0.623226
1  0.701131         0.644899      0.639442  0.641792
2  0.693053         0.621276   

# ✅**Final Submission**

In [21]:
train_df = pd.read_csv(settings.TRAIN_PATH)
test_df  = pd.read_csv(settings.TEST_PATH)


X_train_full = train_df.drop(columns=[settings.TARGET])
y_train_full = label_encoder.transform(train_df[settings.TARGET].values)

X_train_full_proc = preprocessor.fit_transform(X_train_full).astype(float)

# final fit
model.fit(X_train_full_proc, y_train_full)

print("Final model trained.")

X_test_proc = preprocessor.transform(test_df).astype(float)

test_preds_int = model.predict(X_test_proc)
test_preds = label_encoder.inverse_transform(test_preds_int)

submission = pd.read_csv(settings.SUBMISSION_PATH)

submission[settings.TARGET] = test_preds
submission.to_csv("final_submission.csv", index=False)

print("\nSubmission saved to final_submission.csv")



Final model trained.

Submission saved to final_submission.csv
